# Descriptive Stats for Crime Reports

In [23]:
import pandas as pd
import numpy as np
import json

In [3]:
data = pd.read_csv('crime_data.csv')

/var/folders/1z/z4nfsvpx2177t7xs71wckg2c7zwknv/T/ipykernel_57713/1350885223.py:1: DtypeWarning: Columns (16,18,19,20) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv('crime_data.csv')


In [7]:
# get list of columns
data.columns

Index(['Unnamed: 0', 'CRIME REF', 'DATE EARLIEST COMMITTED',
       'DATE LATEST COMMITTED', 'DATE FIRST CRIMED', 'Reclassification',
       'OFFENCE FORMER HO CODE', 'OFFENCE RECORDED HO CODE',
       'CRIMSEC3 CATEGORY RECORDED', 'OFFENCE RECORDED GROUP',
       'OFFENCE RECORDED', 'IND RECORDED CRIME', 'V NOMINAL REF',
       'VICTIM CRIME AGE', 'VICTIM GENDER', 'V ETHNIC SELF', 'O NOMINAL REF',
       'O OFFENDER CRIME AGE', 'O OFFENDER GENDER', 'O ETHNIC SELF',
       'O IND ARRESTED', 'VIC OFFENDER RELATIONSHIP', 'IND ALCOHOL',
       'IND DOMESTIC', 'IND DRUGS', 'IND KNIFE INVOLVED', 'IND GUN INVOLVED',
       'DISPOSAL TEXT', 'EVENT TYPE', 'Any Arrest', 'Risk', 'CRIME NOTES'],
      dtype='object')

In [16]:
# check some of the columns for content
data.head()

,Unnamed: 0,CRIME REF,DATE EARLIEST COMMITTED,DATE LATEST COMMITTED,DATE FIRST CRIMED,Reclassification,OFFENCE FORMER HO CODE,OFFENCE RECORDED HO CODE,CRIMSEC3 CATEGORY RECORDED,OFFENCE RECORDED GROUP,...,IND ALCOHOL,IND DOMESTIC,IND DRUGS,IND KNIFE INVOLVED,IND GUN INVOLVED,DISPOSAL TEXT,EVENT TYPE,Any Arrest,Risk,CRIME NOTES
0,NaN,516162841.0,30/04/2016 17:52:00,01/05/2016 17:52:00,18/07/2016 17:53:46,NaN,2811.0,2811.0,BURGLARY,Burglary Dwelling,...,N,Y,N,N,N,"15: CPS - NAMED SUSPECT, VICTIM SUPPORTS BUT E...",Domestic abuse - without children,N,DA risk-Bronze,"THE VICTIM AND SUSPECT ARE BROTHER AND SISTER,..."
1,NaN,512097868.0,19/06/2012 03:30:00,19/06/2012 03:30:00,28/11/2016 12:47:24,NaN,5800.0,5800.0,CRIMINAL DAMAGE,Criminal Damage excluding Arson,...,Y,Y,N,N,N,"15: CPS - NAMED SUSPECT, VICTIM SUPPORTS BUT E...",Criminal damage,Y,NaN,NaN
2,NaN,517121100.0,17/05/2017 14:00:00,17/05/2017 14:00:00,17/05/2017 19:24:57,Y,10501.0,806.0,VIOLENCE,Violence With Injury,...,N,Y,N,N,N,16: VICTIM DECLINES/WITHDRAWS SUPPORT - NAMED ...,Violence against the person,N,NaN,MARK PUSHES ZOE TO WAKE HER UP AND THIS RESULT...
3,NaN,517121132.0,16/05/2017 22:00:00,17/05/2017 09:00:00,18/05/2017 15:30:33,NaN,5803.0,5803.0,CRIMINAL DAMAGE,Criminal Damage excluding Arson,...,N,Y,N,N,N,16: VICTIM DECLINES/WITHDRAWS SUPPORT - NAMED ...,Domestic abuse - with children,N,DA risk-Bronze,THIRD HAND REPORT OF DOMESTIC ASSAULT. PATROL ...
4,NaN,517137157.0,04/06/2017 06:40:00,04/06/2017 06:40:00,04/06/2017 10:29:01,NaN,801.0,801.0,VIOLENCE,Violence With Injury,...,Y,Y,Y,N,N,1: CHARGED,Violence against the person,Y,NaN,KNOWN OFFENDER CAUSED BROKEN BONES AND FACIAL ...


In [21]:
# standardize NaNs
nan_replace_value = 'missing_value'

data = data.fillna(nan_replace_value)

In [22]:
# print the set of values in each of the columns we are interested in in order to see which have high numbers of unique values and may require remapping
columns = [
    'CRIMSEC3 CATEGORY RECORDED',
    'VICTIM CRIME AGE',
    'V ETHNIC SELF',
    'O OFFENDER CRIME AGE',
    'O ETHNIC SELF',
    'VIC OFFENDER RELATIONSHIP'
]

for col in columns:
    print(col)
    print(set(data[col]))
    print('------------------')

CRIMSEC3 CATEGORY RECORDED
{'missing_value', 'DRUGS', 'ROBBERY', 'BURGLARY', 'CRIMINAL DAMAGE', 'VIOLENCE', 'PUBLIC ORDER', 'VEHICLE OFFENCES', 'OTHER', 'THEFT', 'POSSESSION OF WEAPON', 'SEXUAL'}
------------------
VICTIM CRIME AGE
{0.0, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0, 13.0, 14.0, 15.0, 16.0, 17.0, 18.0, 19.0, 20.0, 21.0, 22.0, 23.0, 24.0, 25.0, 26.0, 27.0, 28.0, 29.0, 30.0, 31.0, 32.0, 33.0, 34.0, 35.0, 36.0, 37.0, 38.0, 39.0, 40.0, 41.0, 42.0, 43.0, 44.0, 45.0, 46.0, 47.0, 48.0, 49.0, 50.0, 51.0, 52.0, 53.0, 54.0, 55.0, 56.0, 57.0, 58.0, 59.0, 60.0, 61.0, 62.0, 63.0, 64.0, 65.0, 66.0, 67.0, 68.0, 69.0, 70.0, 71.0, 72.0, 73.0, 74.0, 75.0, 76.0, 77.0, 78.0, 79.0, 80.0, 81.0, 82.0, 83.0, 84.0, 85.0, 86.0, 87.0, 88.0, 89.0, 90.0, 91.0, 92.0, 93.0, 94.0, 95.0, 'missing_value'}
------------------
V ETHNIC SELF
{'A1. ASIAN - INDIAN', 'O9. ANY OTHER ETHNIC GROUP', 'B2. BLACK AFRICAN', 'M2. WHITE & BLACK AFRICAN', 'M9. ANY OTHER MIXED BACKGROUND', 'A2. ASIAN - P

In [38]:
# remap categories based on json file
with open('mappings.json','r') as file:
    mappings = json.load(file)

for col, mapping in mappings.items():
    data[col] = data[col].replace(mapping)

In [39]:
# function to remap numerical categories (i.e., age)
def age_group(age):
    if isinstance(age, (int, float)):
        if age <20:
            return '<20'
        elif 20 <= age < 30:
            return '20-29'
        elif 30 <= age < 40:
            return '30-39'
        elif 40 <= age < 50:
            return '40-49'
        elif 50 <= age < 60:
            return '50-59'
        elif 60 <= age < 70:
            return '60-69'
        elif 70 <= age < 80:
            return '70-79'
        elif 80 <= age:
            return '80+'
        else:
            return 'nan/missing/other'
    else:
        return 'nan/missing/other'

data['v_age_category'] = data['VICTIM CRIME AGE'].apply(age_group)
data['o_age_category'] = data['O OFFENDER CRIME AGE'].apply(age_group)

In [47]:
# function for calculating confidence intervals
def confidence_intervals(
        col_name: str, 
        data: pd.DataFrame,
        z: float = 1.96 # for 95% CI
        ):

    # number of data points
    n = len(data)

    # find proportions of each category
    proportions = data[col_name].value_counts(normalize=True)

    # dataframe to store CIs and percentages
    ci_df = pd.DataFrame(index=proportions.index, columns=['Percentage','Lower CI','Upper CI'])

    for value in proportions.index:
        p = proportions[value]
        SE = np.sqrt(p * (1 - p) / n)
        ci_df.loc[value, 'Percentage'] = round(p * 100, 1)
        ci_df.loc[value, 'Lower CI'] = round((p - z * SE) * 100, 1)
        ci_df.loc[value, 'Upper CI'] = round((p + z * SE) * 100, 1)

    return ci_df

In [49]:
'''
COLUMNS WE'RE INTERESTED IN:
    'CRIMSEC3 CATEGORY RECORDED',
    'v_age_category',
    'V ETHNIC SELF',
    'o_age_category',
    'O ETHNIC SELF',
    'VIC OFFENDER RELATIONSHIP'
    '''

print(
    confidence_intervals(
        data=data,
        col_name='O ETHNIC SELF'
    )
)

              Percentage Lower CI Upper CI
O ETHNIC SELF                             
white               70.5     69.9     71.1
missing_value       25.1     24.5     25.7
black                1.8      1.6      2.0
mixed                1.2      1.0      1.3
asian                1.0      0.8      1.1
other                0.5      0.4      0.6
